# Macro Surprise + VWAP Execution : premier screening actions US

*Premiere tentative pour tester si les journees de surprises macroeconomiques peuvent creer des patterns court terme sur une selection d'actions US.*

Ce notebook est le resume lisible du projet. Les scripts Python restent le moteur du backtest. Ici, je charge seulement les CSV et les figures deja generes, puis j'explique ce qui a ete teste, ce que les tests de robustesse veulent dire, et les limites a garder en tete.

## Idee De Recherche

Je voulais tester si les surprises macroeconomiques peuvent aider a reperer des opportunites court terme sur des actions individuelles.

L'idee n'est pas simplement d'acheter les bonnes nouvelles macro et de vendre les mauvaises. Le but est de combiner un signal de surprise macro avec une regle d'execution intraday plus disciplinee autour du VWAP.

Pour cette version US, il faut etre precis dans l'interpretation. Beaucoup de publications macro importantes sortent avant l'ouverture des actions US. Donc la version actuelle doit plutot etre lue comme :

**jour de macro + VWAP intraday**

plutot que :

**reaction pure minute par minute au timestamp de publication**

Cette nuance est importante. Si je montre ce travail a un trader, je ne veux pas pretendre mesurer une reaction parfaite a la seconde de publication alors que l'entree se fait souvent a l'ouverture ou apres.

## Donnees Utilisees

L'etude combine deux types de donnees :

- publications macroeconomiques US, avec actual, estimate et previous quand disponible ;
- metriques historiques de surprise par type d'evenement, utilisees pour normaliser la surprise ;
- donnees OHLCV minute sur les actions ;
- VWAP intraday calcule a partir des barres minute, avec reset chaque jour.

Pour ce premier screening, j'ai travaille avec environ 10 ans de donnees en timeframe 1 minute sur les actions testees. Les fichiers minute sont lourds et je n'arrive pas a les charger correctement ici, donc ils ne sont pas inclus dans le GitHub.

Si tu lis ce repo et que tu veux verifier les fichiers exacts utilises pour reproduire le test, contacte-moi et je peux te les envoyer par mail.

Les donnees brutes ne sont pas incluses dans ce repo. Le notebook attend seulement des CSV de synthese et des figures de rapport.

## Construction Du Signal

La surprise macro est construite en trois etapes :

```text
surprise_raw = actual - estimate
surprise_z = (surprise_raw - historical_average_surprise) / historical_surprise_std
signal = surprise_z * direction
```

`direction = +1` quand une valeur macro plus elevee est consideree comme positive.

`direction = -1` quand une valeur macro plus faible est consideree comme positive.

Les evenements marques `MIXED` sont exclus parce que le signe n'est pas assez clair.

Le moteur teste ensuite deux hypotheses :

- `DRIFT` : trader dans le sens du signal macro.
- `FADE` : trader contre le signal macro.

## Logique D'execution VWAP

La regle VWAP est utilisee comme filtre d'execution, pas comme signal macro en soi.

Pour un trade long, le modele prefere entrer quand le prix est sous ou proche du VWAP. Pour un short, il prefere entrer quand le prix est au-dessus ou proche du VWAP. Si le prix n'est pas favorable, le modele attend un retour vers le VWAP dans une fenetre definie.

Deux modes d'execution sont importants :

- `close_cross` : plus optimiste, car il suffit que la cloture de la bougie respecte la condition.
- `limit_touch` : plus conservateur, car il demande une logique de touch/fill plus proche d'un ordre limite autour du VWAP.

Le VWAP est remis a zero chaque jour. C'est logique pour de l'execution intraday, mais ca veut aussi dire que la strategie peut devenir en partie un test de mean reversion autour du VWAP si je ne fais pas attention. C'est pour ca que les tests random et train/test sont importants.

## Cadre De Robustesse

Le modele ne garde pas une action seulement parce que le backtest complet est profitable.

Les tests de robustesse sont :

- test avec sens de trade aleatoire ;
- test avec timestamp aleatoire ;
- placebo avec decalage temporel ;
- sweep de couts de transaction ;
- split chronologique train/test ;
- walk-forward par annee.

Le split train/test est particulierement important. La regle est selectionnee sur les premiers 70% de la periode, puis testee sur les derniers 30%. Si la regle parait bonne sur l'echantillon complet mais perd de l'argent sur la periode de test, je la rejette ou je la considere comme fragile.

## Chargement Des Sorties

Le notebook attend les fichiers generes dans :

```text
../reports/trader_report/csv_outputs/
```

Il cherche aussi les graphiques PNG dans :

```text
../reports/trader_report/figures/
```

Si un fichier manque, le notebook affiche un avertissement clair et continue.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ModuleNotFoundError:
    plt = None
    HAS_MATPLOTLIB = False

try:
    from IPython.display import display, Image, Markdown
except ModuleNotFoundError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text
    class Image:
        def __init__(self, filename=None, **kwargs):
            self.filename = filename
        def __repr__(self):
            return f"Image({self.filename})"

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "notebooks").exists() and (PROJECT_ROOT.parent / "notebooks").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_DIR = PROJECT_ROOT / "reports" / "trader_report" / "csv_outputs"
FIG_DIR = PROJECT_ROOT / "reports" / "trader_report" / "figures"
NOTES_DIR = PROJECT_ROOT / "reports" / "trader_report" / "notes"

FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTES_DIR.mkdir(parents=True, exist_ok=True)

expected_files = {
    "master": "single_country_master_summary.csv",
    "all_configs": "single_country_all_configs_summary.csv",
    "random_tests": "single_country_random_tests.csv",
    "train_test": "single_country_train_test.csv",
    "walk_forward": "single_country_walk_forward.csv",
}

loaded = {}
missing = []

for key, filename in expected_files.items():
    path = CSV_DIR / filename
    if path.exists():
        loaded[key] = pd.read_csv(path)
        print(f"Charge {filename} : {loaded[key].shape[0]} lignes, {loaded[key].shape[1]} colonnes")
    else:
        missing.append(filename)
        print(f"AVERTISSEMENT : fichier manquant {filename}")

print("\nDossier CSV :", CSV_DIR)
print("Dossier figures :", FIG_DIR)

if missing:
    display(Markdown("**Fichiers manquants :** " + ", ".join(f"`{m}`" for m in missing)))
else:
    display(Markdown("Tous les CSV attendus ont ete charges."))

master = loaded.get("master")
all_configs = loaded.get("all_configs")
random_tests = loaded.get("random_tests")
train_test = loaded.get("train_test")
walk_forward = loaded.get("walk_forward")

## Controles Sur Les Tickers

La presentation des tickers peut sembler un detail, mais ca peut creer de la confusion pendant une revue.

Je veux surtout reperer deux cas :

- un ticker affiche comme `LONDON-STRATEGIC-EDGE`, qui est probablement un probleme d'extraction depuis le nom du fichier ;
- des tickers dupliques, par exemple si le meme CSV brut est present deux fois.

Le code ci-dessous ne modifie pas le resultat du backtest. Il signale seulement les problemes de presentation.

In [ ]:
def extract_ticker_from_filename(name):
    base = Path(name).stem
    base = re.sub(r"\s*\(\d+\)$", "", base)
    m = re.match(r"([A-Za-z]+)_dataset", base)
    if m:
        return m.group(1).upper()
    first = re.split(r"[_\-\s]", base)[0]
    return first.upper() if first else base.upper()

notes = []

if master is not None and "ticker" in master.columns:
    tickers = master["ticker"].astype(str)
    bad = tickers.str.upper().eq("LONDON-STRATEGIC-EDGE")
    if bad.any():
        notes.append("`LONDON-STRATEGIC-EDGE` apparait comme ticker dans le resume. Je le traiterais comme un probleme de presentation, probablement cause par l'extraction du nom de fichier. Si le fichier source correspond a LMB, je l'afficherais comme LMB dans les tableaux sans changer le resultat brut.")

    dupes = tickers[tickers.duplicated(keep=False)].sort_values().unique().tolist()
    if dupes:
        notes.append("Tickers dupliques dans le master summary : " + ", ".join(f"`{d}`" for d in dupes))

raw_dir = PROJECT_ROOT / "US-Actions"
if raw_dir.exists():
    raw_files = sorted(raw_dir.glob("*.csv"))
    raw_tickers = [extract_ticker_from_filename(p.name) for p in raw_files]
    raw_dupes = sorted({t for t in raw_tickers if raw_tickers.count(t) > 1})
    if raw_dupes:
        notes.append("Des fichiers bruts semblent dupliques localement : " + ", ".join(f"`{d}`" for d in raw_dupes) + ". Je les nettoierais avant un run final.")
    if any(t == "LMB" for t in raw_tickers):
        notes.append("Un fichier source `LMB` existe localement. Si un resultat affiche `LONDON-STRATEGIC-EDGE`, je l'afficherais comme `LMB (?)` et j'ajouterais une note sur le probleme de nom de fichier.")

if notes:
    display(Markdown("\n\n".join("- " + n for n in notes)))
else:
    display(Markdown("Aucun probleme de ticker detecte dans les resumes charges ou les noms de fichiers locaux."))

## Vue D'ensemble Des Resultats

Cette section utilise `single_country_master_summary.csv`. Elle donne la vue globale : combien d'actions ont ete testees, comment les classifications se repartissent, quels noms ressortent le mieux par score de confiance, et quels noms sont rejetes.

In [ ]:
def display_master_overview(master_df):
    if master_df is None:
        display(Markdown("`single_country_master_summary.csv` manque, donc la vue d'ensemble ne peut pas encore etre construite."))
        return None

    df = master_df.copy()

    if "ticker" in df.columns:
        bad = df["ticker"].astype(str).str.upper().eq("LONDON-STRATEGIC-EDGE")
        if bad.any():
            df.loc[bad, "ticker"] = "LMB (?)"

    n = len(df)
    display(Markdown(f"**Actions dans le master summary :** {n}"))

    if "classification" in df.columns:
        counts = df["classification"].fillna("MISSING").value_counts().rename_axis("classification").reset_index(name="count")
        display(counts)

    table_cols = [
        "ticker",
        "classification",
        "confidence_score",
        "best_hypothese",
        "best_vwap_mode",
        "best_horizon",
        "best_ret_moy_pct",
        "test_ret_moy_pct",
        "random_side_percentile",
        "random_ts_percentile",
        "cost_break_even_bps",
        "main_warning",
    ]
    present_cols = [c for c in table_cols if c in df.columns]

    sort_cols = [c for c in ["confidence_score", "test_ret_moy_pct", "best_ret_moy_pct"] if c in df.columns]
    ranked = df.sort_values(sort_cols, ascending=False) if sort_cols else df

    display(Markdown("**Meilleurs candidats par score de confiance**"))
    display(ranked[present_cols].head(12) if present_cols else ranked.head(12))

    if "classification" in df.columns:
        rejected_mask = df["classification"].astype(str).str.contains("D_REJECT|INSUFFICIENT", case=False, na=False)
        rejected = df.loc[rejected_mask]
        display(Markdown("**Noms rejetes ou avec donnees insuffisantes**"))
        if rejected.empty:
            display(Markdown("Aucune ligne rejetee ou insuffisante dans le master summary charge."))
        else:
            display(rejected[present_cols].head(30) if present_cols else rejected.head(30))

    return df

master_view = display_master_overview(master)

## Figures Existantes Ou Graphiques De Secours

Si des PNG existent deja, je les affiche directement. Sinon, le notebook essaie de generer quelques graphiques simples a partir des CSV disponibles.

Les graphiques de secours restent volontairement sobres : titres lisibles, axes clairs, pas de style inutile.

In [ ]:
def save_current_fig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=140, bbox_inches="tight")
    plt.show()
    print("Sauvegarde", path)
    return path

existing_pngs = sorted(FIG_DIR.glob("*.png"))

if existing_pngs:
    display(Markdown("**Figures de rapport trouvees :**"))
    for path in existing_pngs:
        display(Markdown(f"`{path.name}`"))
        display(Image(filename=str(path)))
elif not HAS_MATPLOTLIB:
    display(Markdown("Aucune figure PNG trouvee, et `matplotlib` n'est pas installe dans cet environnement. Installer les dependances permet de generer les graphiques de secours."))
else:
    display(Markdown("Aucune figure PNG trouvee. Je genere des graphiques de secours quand les donnees le permettent."))

    generated = []

    if master_view is not None and "classification" in master_view.columns:
        counts = master_view["classification"].fillna("MISSING").value_counts()
        fig, ax = plt.subplots(figsize=(8, 4.5))
        counts.plot(kind="bar", ax=ax, color="#4c78a8")
        ax.set_title("Nombre d'actions par classification")
        ax.set_xlabel("Classification")
        ax.set_ylabel("Nombre d'actions")
        ax.tick_params(axis="x", rotation=35)
        generated.append(save_current_fig("01_classification_count.png"))

    if master_view is not None and {"ticker", "confidence_score"}.issubset(master_view.columns):
        top = master_view.sort_values("confidence_score", ascending=False).head(10).copy()
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top["ticker"].astype(str), top["confidence_score"], color="#59a14f")
        ax.invert_yaxis()
        ax.set_title("Top 10 des actions par score de confiance")
        ax.set_xlabel("Score de confiance")
        ax.set_ylabel("Ticker")
        generated.append(save_current_fig("02_top_10_confidence_score.png"))

    if master_view is not None and {"ticker", "best_ret_moy_pct", "test_ret_moy_pct"}.issubset(master_view.columns):
        plot_df = master_view.dropna(subset=["best_ret_moy_pct", "test_ret_moy_pct"]).copy()
        if not plot_df.empty:
            fig, ax = plt.subplots(figsize=(7.5, 5))
            ax.scatter(plot_df["best_ret_moy_pct"], plot_df["test_ret_moy_pct"], color="#4c78a8", alpha=0.8)
            ax.axhline(0, color="#666666", linewidth=1)
            ax.axvline(0, color="#666666", linewidth=1)
            ann = plot_df.sort_values("confidence_score", ascending=False).head(10) if "confidence_score" in plot_df else plot_df.head(10)
            for _, row in ann.iterrows():
                ax.annotate(str(row["ticker"]), (row["best_ret_moy_pct"], row["test_ret_moy_pct"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
            ax.set_title("Edge full-sample vs edge sur periode de test")
            ax.set_xlabel("Rendement moyen du meilleur backtest complet (%)")
            ax.set_ylabel("Rendement moyen sur test chronologique (%)")
            generated.append(save_current_fig("03_full_period_vs_test_edge.png"))

    if master_view is not None and {"ticker", "random_side_percentile", "cost_break_even_bps"}.issubset(master_view.columns):
        plot_df = master_view.dropna(subset=["random_side_percentile", "cost_break_even_bps"]).copy()
        if not plot_df.empty:
            fig, ax = plt.subplots(figsize=(7.5, 5))
            ax.scatter(plot_df["random_side_percentile"], plot_df["cost_break_even_bps"], color="#f28e2b", alpha=0.8)
            ann = plot_df.sort_values("confidence_score", ascending=False).head(10) if "confidence_score" in plot_df else plot_df.head(10)
            for _, row in ann.iterrows():
                ax.annotate(str(row["ticker"]), (row["random_side_percentile"], row["cost_break_even_bps"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
            ax.set_title("Percentile random-side vs cout break-even")
            ax.set_xlabel("Percentile du test random-side")
            ax.set_ylabel("Cout break-even (bps)")
            generated.append(save_current_fig("04_random_side_vs_cost_break_even.png"))

    wf_ratio = None
    if master_view is not None and {"ticker", "walk_forward_positive_ratio"}.issubset(master_view.columns):
        wf_ratio = master_view[["ticker", "walk_forward_positive_ratio"]].dropna().copy()
    elif walk_forward is not None and "ticker" in walk_forward.columns:
        ret_col = next((c for c in ["ret_moy_pct", "ret_moy_%", "best_ret_moy_pct"] if c in walk_forward.columns), None)
        if ret_col:
            wf_ratio = (walk_forward.assign(_positive=walk_forward[ret_col] > 0)
                        .groupby("ticker", as_index=False)["_positive"].mean()
                        .rename(columns={"_positive": "walk_forward_positive_ratio"}))

    if wf_ratio is not None and not wf_ratio.empty:
        top_wf = wf_ratio.sort_values("walk_forward_positive_ratio", ascending=False).head(15)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.barh(top_wf["ticker"].astype(str), top_wf["walk_forward_positive_ratio"], color="#79706e")
        ax.invert_yaxis()
        ax.set_title("Ratio positif en walk-forward par action")
        ax.set_xlabel("Part des periodes walk-forward positives")
        ax.set_ylabel("Ticker")
        generated.append(save_current_fig("05_walk_forward_positive_ratio.png"))

    if not generated:
        display(Markdown("Aucun graphique de secours n'a pu etre genere parce que les CSV necessaires sont absents."))

## Interpretation

Cette cellule ecrit une interpretation courte a partir du master summary charge, pour que les exemples viennent des donnees au lieu d'etre ecrits en dur.

In [ ]:
def names_from(df, mask, limit=5):
    if df is None or "ticker" not in df.columns:
        return []
    sub = df.loc[mask].copy()
    if "confidence_score" in sub.columns:
        sub = sub.sort_values("confidence_score", ascending=False)
    return sub["ticker"].astype(str).head(limit).tolist()

if master_view is None:
    display(Markdown("L'interpretation se mettra a jour quand `single_country_master_summary.csv` sera disponible."))
else:
    df = master_view.copy()
    cls = df["classification"].astype(str) if "classification" in df.columns else pd.Series([""] * len(df), index=df.index)
    a_names = names_from(df, cls.str.startswith("A"))
    rejected_names = names_from(df, cls.str.contains("D_REJECT|INSUFFICIENT", case=False, na=False))
    fragile_names = names_from(df, cls.str.startswith("B") | cls.str.startswith("C"))

    msg = f"Le premier batch US charge contient {len(df)} noms. "
    if a_names:
        msg += "Les candidats A les plus propres sont " + ", ".join(a_names) + ". "
    else:
        msg += "Il n'y a pas de candidat A dans le resume charge. "
    if fragile_names:
        msg += "Un groupe plus large semble plus fragile ou plus dependant du VWAP, par exemple " + ", ".join(fragile_names) + ". "
    if rejected_names:
        msg += "Les noms rejetes ou avec donnees insuffisantes incluent " + ", ".join(rejected_names) + ". "
    msg += "Le point principal n'est pas seulement le rendement full-sample. Si la regle selectionnee sur le train echoue sur la periode de test, je ne veux pas considerer le resultat complet comme fiable."

    display(Markdown(msg))

## Pourquoi Des Noms Profitables Peuvent Etre Rejetes

Une action peut avoir un backtest full-sample positif et etre quand meme rejetee.

Ca arrive quand le modele trouve une configuration profitable sur tout l'historique, mais que la configuration selectionnee sur la premiere partie ne marche pas sur la periode de test plus recente. Dans ce cas, le resultat peut etre sur-optimise, dependant d'un regime precis, ou surtout lie au filtre VWAP plutot qu'au signal macro.

Le tableau ci-dessous cherche les actions ou :

```text
best_ret_moy_pct > 0
test_ret_moy_pct <= 0
```

In [ ]:
if master_view is None:
    display(Markdown("Impossible de construire ce tableau pour l'instant, car le master summary manque."))
elif {"best_ret_moy_pct", "test_ret_moy_pct"}.issubset(master_view.columns):
    rejected_profitable = master_view[
        (pd.to_numeric(master_view["best_ret_moy_pct"], errors="coerce") > 0)
        & (pd.to_numeric(master_view["test_ret_moy_pct"], errors="coerce") <= 0)
    ].copy()

    cols = ["ticker", "best_ret_moy_pct", "train_ret_moy_pct", "test_ret_moy_pct", "classification", "main_warning"]
    cols = [c for c in cols if c in rejected_profitable.columns]

    if rejected_profitable.empty:
        display(Markdown("Aucune ligne chargee ne correspond a ce filtre."))
    else:
        display(rejected_profitable[cols].sort_values("best_ret_moy_pct", ascending=False))
else:
    display(Markdown("Les colonnes necessaires ne sont pas presentes dans le master summary charge."))

## Avertissement Principal

L'avertissement principal est que la configuration US actuelle contient beaucoup d'entrees `PRE_OPEN_SAME_DAY`.

A cause de ca, je ne dois pas presenter ce travail comme une reaction high-frequency pure aux publications macro. L'interpretation la plus honnete est que je teste si les jours de publication macro creent un environnement intraday ou une regle VWAP peut capter des patterns directionnels ou contrariants.

C'est quand meme interessant, mais ce n'est pas la meme affirmation. Avant de traiter un resultat comme une vraie idee de trading, je voudrais des timestamps plus propres, plus de simulations aleatoires, et des hypotheses d'execution plus conservatrices.

## Prochaines Etapes

- Relancer les meilleurs candidats avec plus de simulations random.
- Nettoyer les noms de tickers et supprimer les fichiers bruts dupliques avant le run final.
- Tester `limit_touch` plus en profondeur, parce que c'est plus proche d'une execution realiste.
- Ameliorer la gestion des timestamps, surtout pour les publications US pre-ouverture.
- Tester un univers plus large mais controle.
- Construire une petite version paper-trading seulement apres validation plus stricte des candidats.